# 🧪 End-to-End API Testing — From Startup to Prediction

> **Module 8 of 8** — The Cervical Cancer Prediction Service

---

## 🎯 What This Module Does

This final notebook demonstrates the **complete end-to-end flow** of the Cervical Cancer Prediction Service — from starting the server to making live predictions and generating clinical reports.

## 🏗️ The Complete Pipeline

```
┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐   ┌──────────┐
│  Start   │──▶│  Health  │──▶│  Models  │──▶│ Predict  │──▶│  Report  │
│  Server  │   │  Check   │   │  List    │   │          │   │          │
└──────────┘   └──────────┘   └──────────┘   └──────────┘   └──────────┘
     │              │              │              │              │
     ▼              ▼              ▼              ▼              ▼
  uvicorn       GET /health    GET /models    POST /predict  POST /predict/report
```

## 🧪 What We'll Test

| # | Test | Endpoint | Expected |
|---|------|----------|----------|
| 1 | Server starts | — | Port 8000 listening |
| 2 | Health check | `GET /health` | 200, model loaded, 35 features |
| 3 | List models | `GET /models` | 200, 8 registered models |
| 4 | Predict (default) | `POST /predict` | 200, prediction + confidence |
| 5 | Predict (specific) | `POST /predict?model=Random_Forest` | 200, different model |
| 6 | Generate report | `POST /predict/report` | 200, Markdown report |

---

## 🧠 Skills Demonstrated

- ✅ **Integration Testing** — full-stack API verification
- ✅ **API Testing** — HTTP requests, status codes, JSON parsing
- ✅ **Uvicorn** — ASGI server deployment
- ✅ **End-to-End Validation** — complete pipeline verification

In [ ]:
# ── Add the api directory to sys.path so 'app' module can be found ──
import sys
from pathlib import Path

_candidates = [
    Path.cwd() / "cervical-cancer-mlops" / "api",
    Path.cwd().parent / "api",
    Path.cwd() / "api",
]
api_dir = next((p for p in _candidates if (p / "app").exists()), None)
if api_dir is None:
    raise RuntimeError(
        "Could not locate the 'api/app' package. "
        "Expected it under one of: " + ", ".join(map(str, _candidates))
    )
if str(api_dir) not in sys.path:
    sys.path.insert(0, str(api_dir))
print(f"✅ Added {api_dir} to sys.path")


In [ ]:
# ── Import test utilities ──
import json
import urllib.request
import urllib.error

BASE = "http://127.0.0.1:8000"

def api_get(path):
    """Send a GET request and return (status, body)."""
    req = urllib.request.Request(BASE + path, method="GET")
    try:
        with urllib.request.urlopen(req) as resp:
            return resp.status, json.loads(resp.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read().decode())

def api_post(path, data):
    """Send a POST request and return (status, body)."""
    req = urllib.request.Request(
        BASE + path,
        data=json.dumps(data).encode(),
        headers={"Content-Type": "application/json"},
        method="POST"
    )
    try:
        with urllib.request.urlopen(req) as resp:
            return resp.status, json.loads(resp.read().decode())
    except urllib.error.HTTPError as e:
        return e.code, json.loads(e.read().decode())

print("✅ Test utilities ready")

In [ ]:
# ── Test 1: Health Check ──
code, body = api_get("/health")

print("🏥 TEST 1: HEALTH CHECK")
print("=" * 50)
print(f"Status: {code}")
if code == 200:
    print(f"✅ Service is healthy!")
    print(f"   Model:       {body['model_name']} v{body['model_version']}")
    print(f"   Stage:       {body['model_stage']}")
    print(f"   Features:    {body['features_count']}")
else:
    print(f"❌ Health check failed: {body}")

In [ ]:
# ── Test 2: List Models ──
code, body = api_get("/models")

print("🤖 TEST 2: LIST MODELS")
print("=" * 50)
print(f"Status: {code}")
if code == 200:
    print(f"✅ Found {body['total']} models!")
    print(f"   Default: {body['default_model']}")
    print()
    for m in body['models']:
        print(f"   - {m['name']:<25} v{m['version']} ({m['stage']})")
else:
    print(f"❌ Failed to list models: {body}")

In [ ]:
# ── Test 3: Predict with default model ──
patient_data = {
    "age": 25, "number_of_sexual_partners": 1, "first_sexual_intercourse": 22,
    "num_of_pregnancies": 0, "smokes": 0, "smokes_years": 0,
    "smokes_packs_per_year": 0.0, "hormonal_contraceptives": 0,
    "hormonal_contraceptives_years": 0.0, "iud": 0, "iud_years": 0.0,
    "stds": 0, "stds_number": 0, "stds_condylomatosis": 0,
    "stds_cervical_condylomatosis": 0, "stds_vaginal_condylomatosis": 0,
    "stds_vulvo_perineal_condylomatosis": 0, "stds_syphilis": 0,
    "stds_pelvic_inflammatory_disease": 0, "stds_genital_herpes": 0,
    "stds_molluscum_contagiosum": 0, "stds_aids": 0, "stds_hiv": 0,
    "stds_hepatitis_b": 0, "stds_hpv": 0, "stds_number_of_diagnosis": 0,
    "stds_time_since_first_diagnosis": 0.0, "stds_time_since_last_diagnosis": 0.0,
    "dx_cancer": 0, "dx_cin": 0, "dx_hpv": 0, "dx": 0,
    "hinselmann": 0, "schiller": 0, "citology": 0
}

code, body = api_post("/predict", patient_data)

print("🔮 TEST 3: PREDICT (DEFAULT MODEL)")
print("=" * 50)
print(f"Status: {code}")
if code == 200:
    print(f"✅ Prediction successful!")
    print(f"   Model:       {body['model_name']} v{body['model_version']}")
    print(f"   Prediction:  {body['prediction_label']} ({body['prediction']})")
    print(f"   Confidence:  {body['confidence']:.4f}")
    print(f"   Threshold:   {body['threshold']}")
else:
    print(f"❌ Prediction failed: {body}")

In [ ]:
# ── Test 4: Predict with a specific model ──
code, body = api_post("/predict?model=Random_Forest", patient_data)

print("🔄 TEST 4: PREDICT (RANDOM_FOREST)")
print("=" * 50)
print(f"Status: {code}")
if code == 200:
    print(f"✅ Dynamic model switching works!")
    print(f"   Model:       {body['model_name']} v{body['model_version']}")
    print(f"   Prediction:  {body['prediction_label']} ({body['prediction']})")
    print(f"   Confidence:  {body['confidence']:.4f}")
else:
    print(f"❌ Prediction failed: {body}")

In [ ]:
# ── Test 5: Generate clinical report ──
code, body = api_post("/predict/report", patient_data)

print("📄 TEST 5: GENERATE CLINICAL REPORT")
print("=" * 50)
print(f"Status: {code}")
if code == 200:
    print(f"✅ Report generated!")
    print(f"   Model:       {body['model_name']} v{body['model_version']}")
    print(f"   Prediction:  {body['prediction_label']} ({body['prediction']})")
    print(f"   Risk Level:  {body['risk_level']}")
    print(f"   Report:      {len(body['report_markdown'])} chars of Markdown")
    print()
    print("   Preview:")
    for line in body['report_markdown'].split("\n")[:8]:
        print(f"   {line}")
else:
    print(f"❌ Report generation failed: {body}")

In [ ]:
# ── Test 6: Invalid input rejected ──
bad_data = dict(patient_data)
bad_data["age"] = -5  # Invalid age

code, body = api_post("/predict", bad_data)

print("🚨 TEST 6: INVALID INPUT REJECTED")
print("=" * 50)
print(f"Status: {code}")
if code == 422:
    print(f"✅ Validation correctly rejected invalid input!")
    print(f"   Error: {body.get('error', 'Validation Error')}")
else:
    print(f"⚠️ Expected 422, got {code}: {body}")

---

## 🎉 Final Summary

The **Cervical Cancer Prediction Service** is fully operational:

| Capability | Status |
|------------|--------|
| ✅ Centralized configuration | Pydantic Settings |
| ✅ Strict data validation | 35-feature Pydantic schema |
| ✅ Correct preprocessing | Feature ordering guaranteed |
| ✅ MLflow model registry | 8 versioned models |
| ✅ REST API | Health, predict, report endpoints |
| ✅ Clinical reports | Explainable AI for healthcare |
| ✅ Error handling | Structured 422/500 responses |

## 🚀 How to Run

```bash
cd cervical-cancer-mlops/api
python run.py
```

Then open:
- **Swagger UI:** http://localhost:8000/docs
- **Health check:** http://localhost:8000/health
- **Prediction:** POST http://localhost:8000/predict
- **Report:** POST http://localhost:8000/predict/report

---

## 🧠 Complete Skills Portfolio

- ✅ **Python** — FastAPI, Pydantic, NumPy, MLflow
- ✅ **MLOps** — Model Registry, versioning, staging, deployment
- ✅ **REST API** — design, testing, Swagger/OpenAPI
- ✅ **Data Validation** — Pydantic v2, type safety
- ✅ **Explainable AI** — clinical decision support reports
- ✅ **Software Engineering** — modular architecture, error handling, testing